Imports

In [1]:
import pandas as pd

Load Data

In [2]:
finviz_data = pd.read_csv('Data/finviz_news.csv')
finnhub_data = pd.read_csv('Data/finnhub_news.csv')
gsib_data = pd.read_csv('Data/GSIB_full_News.csv')

Inspect Data

In [3]:
finviz_data.head()
finnhub_data.head()
gsib_data.head()

,date,ticker_symbol,title,link,sentiment
0,2016-02-29T15:07:00+00:00,JPM,JPMorgan Chase and Syracuse University Renew P...,https://www.globenewswire.com/news-release/201...,"{'polarity': 0.998, 'neg': 0.007, 'neu': 0.877..."
1,2016-04-28T11:30:00+00:00,JPM,"Cardtronics buys 2,586 off-premise ATMs from C...",https://www.globenewswire.com/news-release/201...,"{'polarity': 0.881, 'neg': 0, 'neu': 0.954, 'p..."
2,2016-04-21T14:06:00+00:00,JPM,Grameen Foundation and ideas42 Launch Partners...,https://www.globenewswire.com/news-release/201...,"{'polarity': 0.998, 'neg': 0.02, 'neu': 0.86, ..."
3,2016-03-22T14:00:00+00:00,JPM,Veteran Launch Addresses Unmet Financing Needs...,https://www.globenewswire.com/news-release/201...,"{'polarity': 0.998, 'neg': 0.015, 'neu': 0.824..."
4,2016-03-10T18:17:00+00:00,JPM,The Madison Square Garden Company Announces t...,https://www.globenewswire.com/news-release/201...,"{'polarity': 0.999, 'neg': 0.012, 'neu': 0.883..."


Clean Data

In [4]:
import re
from ast import literal_eval

def parse_finviz_cell(raw_value):
    if pd.isna(raw_value):
        return pd.Series({'date': pd.NaT, 'headline': pd.NA})
    text = str(raw_value).strip()
    try:
        parsed = literal_eval(text)
        if isinstance(parsed, tuple) and len(parsed) >= 2:
            raw_date, raw_headline = parsed[0], parsed[1]
            return pd.Series({
                'date': pd.to_datetime(raw_date, errors='coerce'),
                'headline': str(raw_headline).strip() if raw_headline is not None else pd.NA
            })
    except Exception:
        pass
    date_match = re.search(r"\(\s*'([^']+)'\s*,", text)
    head_match = re.search(r"\(\s*'[^']+'\s*,\s*'(.+?)'\s*,\s*'https?://", text)
    fallback_date = pd.to_datetime(date_match.group(1), errors='coerce') if date_match else pd.NaT
    fallback_headline = head_match.group(1).strip() if head_match else pd.NA
    return pd.Series({'date': fallback_date, 'headline': fallback_headline})

finviz_long = finviz_data.melt(var_name='stock', value_name='raw_news').dropna(subset=['raw_news'])
parsed = finviz_long['raw_news'].apply(parse_finviz_cell)
finviz_clean = pd.concat([finviz_long[['stock']], parsed], axis=1)

finnhub_clean = finnhub_data.rename(columns={'ticker_symbol': 'stock'})[['stock', 'headline', 'date']].copy()
finnhub_clean['date'] = pd.to_datetime(finnhub_clean['date'], errors='coerce')
finnhub_clean['headline'] = finnhub_clean['headline'].astype(str).str.strip()

# GSIB data standardization (different column names: ticker_symbol, title)
gsib_clean = gsib_data.rename(columns={'ticker_symbol': 'stock', 'title': 'headline'})[['stock', 'headline', 'date']].copy()
gsib_clean['date'] = pd.to_datetime(gsib_clean['date'], errors='coerce')
gsib_clean['headline'] = gsib_clean['headline'].astype(str).str.strip()

for df in (finviz_clean, finnhub_clean, gsib_clean):
    df['stock'] = df['stock'].astype(str).str.upper().str.strip()
    df['headline'] = df['headline'].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

finviz_clean = finviz_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finviz_clean['date_only'] = finviz_clean['date'].dt.date
finnhub_clean = finnhub_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finnhub_clean['date_only'] = finnhub_clean['date'].dt.date
gsib_clean = gsib_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
gsib_clean['date_only'] = gsib_clean['date'].dt.date

Save Cleaned Data

In [5]:
finviz_clean.to_csv('Data/finviz_clean.csv', index=False)
finnhub_clean.to_csv('Data/finnhub_clean.csv', index=False)
gsib_clean.to_csv('Data/GSIB_clean.csv', index=False)